In [0]:
from pyspark.sql import functions as F

1. Read Data (with inferSchema)


In [0]:
df1 = spark.read.option("sep", ";").csv("/FileStore/tables/vendor1.csv", inferSchema=True, header=True)
df2 = spark.read.option("multiLine", True).json("/FileStore/tables/vendor2.json")
df3 = spark.read.parquet("/FileStore/tables/.vendor3.parquet")

2. Standardize Schema

In [0]:
df1 = df1.withColumnRenamed("product_id", "product_id").withColumn("data_source", F.lit("vendor1"))
df2 = df2.withColumnRenamed("item_code", "product_id").withColumn("data_source", F.lit("vendor2"))
df3 = df3.withColumnRenamed("prod_id", "product_id").withColumn("data_source", F.lit("vendor3"))

                                                                                        

3. Select Required Columns

In [0]:
cols = ["date", "product_id", "total_units", "total_revenue","data_source"]
df1 = df1.select(cols)
df2 = df2.select(cols)
df3 = df3.select(cols)


4. Combine Data

In [0]:
df = df1.unionByName(df2, allowMissingColumns=True).unionByName(df3, allowMissingColumns=True)

5. Data Cleaning & Casting

In [0]:
df = df.fillna({"total_units": 0, "total_revenue": 0})
df = df.withColumn("date", F.to_date("date"))

6. Validation & Dedup

In [0]:
df = df.filter(F.col("product_id").isNotNull()).filter(F.col("date").isNotNull()).filter(F.col("total_units") >= 0)
df = df.dropDuplicates(["date", "product_id", "data_source"])

7. Aggregation

In [0]:
final_df = df.groupBy("date", "product_id", "data_source").agg(F.sum("total_units").alias("total_units"), F.sum("total_revenue").alias("total_revenue"))
final_df = final_df.orderBy("date", "product_id")


 8. Write Output

In [0]:
final_df.write.mode("overwrite").partitionBy("date", "product_id").parquet("output/final_sales")